# Database Connection Test - Aurora & TiDB

This notebook tests connections to both database systems:
- **Aurora RDS MySQL**: Non-partitioned OLAP/Analytics database
- **TiDB**: Partitioned OLTP/Transactional database

In [3]:
# Import required libraries
import pymysql
import os
from dotenv import load_dotenv
from datetime import datetime

## Load Environment Variables

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Aurora RDS MySQL Configuration
AURORA_HOST = os.getenv('AURORA_HOST')
AURORA_PORT = int(os.getenv('AURORA_PORT', 3306))
AURORA_USER = os.getenv('AURORA_USER', 'admin')
AURORA_PASSWORD = os.getenv('AURORA_PASSWORD')
AURORA_DATABASE = os.getenv('AURORA_DATABASE', 'ai_state_management')

# TiDB Configuration
TIDB_HOST = os.getenv('TIDB_HOST', '127.0.0.1')
TIDB_PORT = int(os.getenv('TIDB_PORT', 3306))
TIDB_USER = os.getenv('TIDB_USER', 'root')
TIDB_PASSWORD = os.getenv('TIDB_PASSWORD', '')
TIDB_DATABASE = os.getenv('TIDB_DATABASE', 'ai_state_management')

print("Aurora Configuration:")
print(f"  Host: {AURORA_HOST}")
print(f"  Port: {AURORA_PORT}")
print(f"  User: {AURORA_USER}")
print(f"  Database: {AURORA_DATABASE}")

print("\nTiDB Configuration:")
print(f"  Host: {TIDB_HOST}")
print(f"  Port: {TIDB_PORT}")
print(f"  User: {TIDB_USER}")
print(f"  Database: {TIDB_DATABASE}")

print("\n✓ Environment variables loaded")

Aurora Host: ai-memory.chc6sicssfys.us-east-2.rds.amazonaws.com
Aurora Port: 3306
Aurora User: admin
Aurora Database: ai_state_management
✓ Aurora environment variables loaded


---

# Part 1: Aurora RDS MySQL Testing

## Connect to Aurora

In [ ]:
# Connect to Aurora RDS MySQL
try:
    aurora_connection = pymysql.connect(
        host=AURORA_HOST,
        port=AURORA_PORT,
        user=AURORA_USER,
        password=AURORA_PASSWORD,
        database=AURORA_DATABASE,
        cursorclass=pymysql.cursors.DictCursor,
        connect_timeout=10
    )
    print(f"✓ Successfully connected to Aurora RDS MySQL")
    print(f"  Database: {AURORA_DATABASE}")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    raise

✓ Successfully connected to Aurora RDS MySQL
  Database: ai_state_management


## Aurora Basic Queries

In [ ]:
# Get MySQL version
with aurora_connection.cursor() as cursor:
    cursor.execute("SELECT VERSION() as version")
    result = cursor.fetchone()
    print(f"MySQL Version: {result['version']}")

MySQL Version: 8.4.7


In [ ]:
# Show current database
with aurora_connection.cursor() as cursor:
    cursor.execute("SELECT DATABASE() as current_db")
    result = cursor.fetchone()
    print(f"Current Database: {result['current_db']}")

Current Database: ai_state_management


In [ ]:
# Show all databases
with aurora_connection.cursor() as cursor:
    cursor.execute("SHOW DATABASES")
    databases = cursor.fetchall()
    print("Available Databases:")
    for db in databases:
        print(f"  - {db['Database']}")

Available Databases:
  - ai_state_management
  - information_schema
  - mysql
  - performance_schema
  - sys


In [ ]:
# Show tables in current database
with aurora_connection.cursor() as cursor:
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()
    if tables:
        print("Tables in Aurora database:")
        for table in tables:
            table_name = list(table.values())[0]
            print(f"  - {table_name}")
    else:
        print("No tables found in current database")

Tables in current database:
  - bots
  - memory_snapshots
  - messages
  - sessions
  - usage_stats
  - user_preferences
  - users


In [ ]:
# Get connection information
with aurora_connection.cursor() as cursor:
    cursor.execute("SELECT USER() as user, CONNECTION_ID() as connection_id")
    result = cursor.fetchone()
    print(f"Connected as: {result['user']}")
    print(f"Connection ID: {result['connection_id']}")

Connected as: admin@98.43.32.82
Connection ID: 4159


## Aurora Simple Query Test

In [ ]:
# Test a simple SELECT query
with aurora_connection.cursor() as cursor:
    cursor.execute("SELECT 'Hello from Aurora!' as message, NOW() as timestamp")
    result = cursor.fetchone()
    print(f"Message: {result['message']}")
    print(f"Timestamp: {result['timestamp']}")

Message: Hello from Aurora!
Timestamp: 2026-04-02 22:41:58


## Aurora Data Loading Validation

Validate that test data was properly loaded into the Aurora database.

In [ ]:
# Verify we're connected to the correct database
with aurora_connection.cursor() as cursor:
    cursor.execute("SELECT DATABASE() as current_db")
    result = cursor.fetchone()
    print(f"✓ Connected to database: {result['current_db']}")

✓ Connected to database: ai_state_management


In [ ]:
# Check table counts
with aurora_connection.cursor() as cursor:
    tables = ['users', 'bots', 'sessions', 'messages', 'memory_snapshots']
    
    print("Aurora Table Row Counts:")
    print("=" * 50)
    
    aurora_table_counts = {}
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) as count FROM {table}")
        result = cursor.fetchone()
        count = result['count']
        aurora_table_counts[table] = count
        status = "✓" if count > 0 else "✗"
        print(f"{status} {table:20s}: {count:,} rows")
    
    print("\n" + "=" * 50)
    print(f"Total records: {sum(aurora_table_counts.values()):,}")

Table Row Counts:
✓ users               : 100 rows
✓ bots                : 16 rows
✓ sessions            : 258 rows
✓ messages            : 7,832 rows
✓ memory_snapshots    : 1,000 rows

Total records: 9,206


In [ ]:
# Validate referential integrity (foreign keys)
print("Aurora Referential Integrity:")
print("=" * 50)

with aurora_connection.cursor() as cursor:
    # Check sessions reference valid users
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM sessions 
        WHERE user_id NOT IN (SELECT user_id FROM users)
    """)
    orphaned_sessions = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_sessions == 0 else "✗"
    print(f"{status} Sessions with valid user_id: {orphaned_sessions} orphaned")
    
    # Check sessions reference valid bots
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM sessions 
        WHERE bot_id NOT IN (SELECT bot_id FROM bots)
    """)
    orphaned_bots = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_bots == 0 else "✗"
    print(f"{status} Sessions with valid bot_id: {orphaned_bots} orphaned")
    
    # Check messages reference valid sessions
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM messages 
        WHERE session_id NOT IN (SELECT session_id FROM sessions)
    """)
    orphaned_messages = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_messages == 0 else "✗"
    print(f"{status} Messages with valid session_id: {orphaned_messages} orphaned")
    
    # Check memory_snapshots reference valid sessions
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM memory_snapshots 
        WHERE session_id NOT IN (SELECT session_id FROM sessions)
    """)
    orphaned_snapshots = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_snapshots == 0 else "✗"
    print(f"{status} Memory snapshots with valid session_id: {orphaned_snapshots} orphaned")
    
    print("\n" + "=" * 50)
    total_orphaned = orphaned_sessions + orphaned_bots + orphaned_messages + orphaned_snapshots
    if total_orphaned == 0:
        print("✓ All foreign key relationships are valid!")
    else:
        print(f"✗ Found {total_orphaned} referential integrity issues")

Validating Referential Integrity:
✓ Sessions with valid user_id: 0 orphaned
✓ Sessions with valid bot_id: 0 orphaned
✓ Messages with valid session_id: 0 orphaned
✓ Memory snapshots with valid session_id: 0 orphaned

✓ All foreign key relationships are valid!


In [ ]:
# Sample some data to verify content
print("Aurora Sample Data:")
print("=" * 50)

with aurora_connection.cursor() as cursor:
    # Sample users
    cursor.execute("SELECT * FROM users LIMIT 3")
    users = cursor.fetchall()
    print(f"\nUsers (showing 3 of {aurora_table_counts['users']}):")
    for user in users:
        print(f"  - {user['user_id']}: {user['username']} ({user['email']})")
    
    # Sample bots
    cursor.execute("SELECT * FROM bots LIMIT 3")
    bots = cursor.fetchall()
    print(f"\nBots (showing 3 of {aurora_table_counts['bots']}):")
    for bot in bots:
        print(f"  - {bot['bot_id']}: {bot['bot_name']} ({bot['bot_type']})")
    
    # Sample sessions
    cursor.execute("SELECT * FROM sessions LIMIT 3")
    sessions = cursor.fetchall()
    print(f"\nSessions (showing 3 of {aurora_table_counts['sessions']}):")
    for session in sessions:
        created = session.get('created_at') or session.get('created') or 'N/A'
        print(f"  - {session['session_id']}: user={session['user_id']}, bot={session['bot_id']}, created={created}")
    
    # Sample messages
    cursor.execute("SELECT * FROM messages LIMIT 3")
    messages = cursor.fetchall()
    print(f"\nMessages (showing 3 of {aurora_table_counts['messages']}):")
    for msg in messages:
        content_preview = msg['content'][:50] + "..." if len(msg['content']) > 50 else msg['content']
        print(f"  - {msg['message_id']}: role={msg['role']}, content=\"{content_preview}\"")
    
    # Check memory snapshots have embeddings
    cursor.execute("SELECT COUNT(*) as count FROM memory_snapshots WHERE embedding IS NOT NULL")
    embedded_count = cursor.fetchone()['count']
    print(f"\nMemory Snapshots with embeddings: {embedded_count:,} of {aurora_table_counts['memory_snapshots']:,}")
    
print("\n" + "=" * 50)
print("✓ Aurora data validation complete!")

Sample Data:

Users (showing 3 of 100):
  - 00ba4f32-1044-4d51-9911-a800a6efef6d: grace.garcia958 (grace.garcia958@example.com)
  - 02da9f3a-03b2-45a5-9117-769ab693f1ba: frank.jackson688 (frank.jackson688@example.com)
  - 02fa3bc6-a15a-4cb0-9997-25bf9c36081b: ruby.smith185 (ruby.smith185@example.com)

Bots (showing 3 of 16):
  - assistant-bot-1: General Assistant (assistant)
  - career-bot-12: Career Counselor (career)
  - coding-bot-9: Code Helper (coding)

Sessions (showing 3 of 258):
  - 00a66426-865c-4798-a772-5f9195a80e10: user=c1f5ff87-9102-427e-94fb-6dfb67ee21b9, bot=career-bot-12, created=N/A
  - 0142fae5-75d6-421f-9b3f-5d350f0c2612: user=86564fe3-c094-4c3e-9be9-a1364122ac7e, bot=assistant-bot-1, created=N/A
  - 026c9cdb-677f-45b2-a08c-defa18ae0807: user=6601abfc-42f0-48a6-92e3-f296a5fdbf21, bot=assistant-bot-1, created=N/A

Messages (showing 3 of 7832):
  - 1: role=user, content="How do I create a to-do list?"
  - 2: role=assistant, content="Here's what I recommend: Would you 

## Close Aurora Connection

In [ ]:
# Close the Aurora connection
aurora_connection.close()
print("✓ Aurora connection closed")

✓ Connection closed


---

# Part 2: TiDB Testing

## Connect to TiDB

In [ ]:
# Connect to TiDB
try:
    tidb_connection = pymysql.connect(
        host=TIDB_HOST,
        port=TIDB_PORT,
        user=TIDB_USER,
        password=TIDB_PASSWORD,
        database=TIDB_DATABASE,
        cursorclass=pymysql.cursors.DictCursor,
        connect_timeout=10
    )
    print(f"✓ Successfully connected to TiDB")
    print(f"  Database: {TIDB_DATABASE}")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("\nMake sure TiDB is running: docker-compose up -d")
    raise

## TiDB Basic Queries

In [ ]:
# Get TiDB version
with tidb_connection.cursor() as cursor:
    cursor.execute("SELECT VERSION() as version")
    result = cursor.fetchone()
    print(f"TiDB Version: {result['version']}")

In [ ]:
# Show current database
with tidb_connection.cursor() as cursor:
    cursor.execute("SELECT DATABASE() as current_db")
    result = cursor.fetchone()
    print(f"Current Database: {result['current_db']}")

In [ ]:
# Show tables in TiDB database
with tidb_connection.cursor() as cursor:
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()
    if tables:
        print("Tables in TiDB database:")
        for table in tables:
            table_name = list(table.values())[0]
            print(f"  - {table_name}")
    else:
        print("No tables found in current database")

In [ ]:
# Check TiDB partition information
with tidb_connection.cursor() as cursor:
    cursor.execute("""
        SELECT TABLE_NAME, PARTITION_NAME, PARTITION_EXPRESSION 
        FROM INFORMATION_SCHEMA.PARTITIONS 
        WHERE TABLE_SCHEMA = %s AND PARTITION_NAME IS NOT NULL
        ORDER BY TABLE_NAME, PARTITION_ORDINAL_POSITION
    """, (TIDB_DATABASE,))
    partitions = cursor.fetchall()
    if partitions:
        print("TiDB Partitioned Tables:")
        current_table = None
        for p in partitions:
            if p['TABLE_NAME'] != current_table:
                current_table = p['TABLE_NAME']
                print(f"\n  {current_table} (partitioned by {p['PARTITION_EXPRESSION']}):")
            print(f"    - {p['PARTITION_NAME']}")
    else:
        print("No partitioned tables found")

## TiDB Data Loading Validation

Validate that test data was properly loaded into the TiDB database.

In [ ]:
# Check table counts
with tidb_connection.cursor() as cursor:
    tables = ['users', 'bots', 'sessions', 'messages', 'memory_snapshots']
    
    print("TiDB Table Row Counts:")
    print("=" * 50)
    
    tidb_table_counts = {}
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) as count FROM {table}")
        result = cursor.fetchone()
        count = result['count']
        tidb_table_counts[table] = count
        status = "✓" if count > 0 else "✗"
        print(f"{status} {table:20s}: {count:,} rows")
    
    print("\n" + "=" * 50)
    print(f"Total records: {sum(tidb_table_counts.values()):,}")

In [ ]:
# Validate referential integrity (foreign keys)
print("TiDB Referential Integrity:")
print("=" * 50)

with tidb_connection.cursor() as cursor:
    # Check sessions reference valid users
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM sessions 
        WHERE user_id NOT IN (SELECT user_id FROM users)
    """)
    orphaned_sessions = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_sessions == 0 else "✗"
    print(f"{status} Sessions with valid user_id: {orphaned_sessions} orphaned")
    
    # Check sessions reference valid bots
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM sessions 
        WHERE bot_id NOT IN (SELECT bot_id FROM bots)
    """)
    orphaned_bots = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_bots == 0 else "✗"
    print(f"{status} Sessions with valid bot_id: {orphaned_bots} orphaned")
    
    # Check messages reference valid sessions
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM messages 
        WHERE session_id NOT IN (SELECT session_id FROM sessions)
    """)
    orphaned_messages = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_messages == 0 else "✗"
    print(f"{status} Messages with valid session_id: {orphaned_messages} orphaned")
    
    # Check memory_snapshots reference valid sessions
    cursor.execute("""
        SELECT COUNT(*) as orphaned 
        FROM memory_snapshots 
        WHERE session_id NOT IN (SELECT session_id FROM sessions)
    """)
    orphaned_snapshots = cursor.fetchone()['orphaned']
    status = "✓" if orphaned_snapshots == 0 else "✗"
    print(f"{status} Memory snapshots with valid session_id: {orphaned_snapshots} orphaned")
    
    print("\n" + "=" * 50)
    total_orphaned = orphaned_sessions + orphaned_bots + orphaned_messages + orphaned_snapshots
    if total_orphaned == 0:
        print("✓ All foreign key relationships are valid!")
    else:
        print(f"✗ Found {total_orphaned} referential integrity issues")

In [ ]:
# Sample some data to verify content
print("TiDB Sample Data:")
print("=" * 50)

with tidb_connection.cursor() as cursor:
    # Sample users
    cursor.execute("SELECT * FROM users LIMIT 3")
    users = cursor.fetchall()
    print(f"\nUsers (showing 3 of {tidb_table_counts['users']}):")
    for user in users:
        print(f"  - {user['user_id']}: {user['username']} ({user['email']})")
    
    # Sample bots
    cursor.execute("SELECT * FROM bots LIMIT 3")
    bots = cursor.fetchall()
    print(f"\nBots (showing 3 of {tidb_table_counts['bots']}):")
    for bot in bots:
        print(f"  - {bot['bot_id']}: {bot['bot_name']} ({bot['bot_type']})")
    
    # Sample sessions
    cursor.execute("SELECT * FROM sessions LIMIT 3")
    sessions = cursor.fetchall()
    print(f"\nSessions (showing 3 of {tidb_table_counts['sessions']}):")
    for session in sessions:
        created = session.get('created_at') or session.get('created') or 'N/A'
        print(f"  - {session['session_id']}: user={session['user_id']}, bot={session['bot_id']}, created={created}")
    
    # Sample messages
    cursor.execute("SELECT * FROM messages LIMIT 3")
    messages = cursor.fetchall()
    print(f"\nMessages (showing 3 of {tidb_table_counts['messages']}):")
    for msg in messages:
        content_preview = msg['content'][:50] + "..." if len(msg['content']) > 50 else msg['content']
        print(f"  - {msg['message_id']}: role={msg['role']}, content=\"{content_preview}\"")
    
    # Check memory snapshots have embeddings
    cursor.execute("SELECT COUNT(*) as count FROM memory_snapshots WHERE embedding IS NOT NULL")
    embedded_count = cursor.fetchone()['count']
    print(f"\nMemory Snapshots with embeddings: {embedded_count:,} of {tidb_table_counts['memory_snapshots']:,}")
    
print("\n" + "=" * 50)
print("✓ TiDB data validation complete!")

## Close TiDB Connection

In [ ]:
# Close the TiDB connection
tidb_connection.close()
print("✓ TiDB connection closed")

---

# Summary

Comparison of both database systems.

In [ ]:
# Compare data between Aurora and TiDB
print("Database Comparison")
print("=" * 70)
print(f"{'Table':<25} {'Aurora':<20} {'TiDB':<20} {'Match':<5}")
print("=" * 70)

tables = ['users', 'bots', 'sessions', 'messages', 'memory_snapshots']
all_match = True

for table in tables:
    aurora_count = aurora_table_counts.get(table, 0)
    tidb_count = tidb_table_counts.get(table, 0)
    match = "✓" if aurora_count == tidb_count else "✗"
    if aurora_count != tidb_count:
        all_match = False
    print(f"{table:<25} {aurora_count:<20,} {tidb_count:<20,} {match:<5}")

print("=" * 70)
aurora_total = sum(aurora_table_counts.values())
tidb_total = sum(tidb_table_counts.values())
print(f"{'TOTAL':<25} {aurora_total:<20,} {tidb_total:<20,} {'✓' if all_match else '✗':<5}")

if all_match:
    print("\n✓ Both databases have identical record counts!")
else:
    print("\n✗ Databases have different record counts")

## Architecture Notes

**Aurora RDS MySQL (`ai_state_management`)**
- Non-partitioned tables
- Optimized for OLAP/Analytics workloads
- Supports complex analytical queries across all data

**TiDB (`ai_state_management`)**
- Partitioned tables (e.g., messages partitioned by session_id)
- Optimized for OLTP/Transactional workloads
- Fast session-based queries with partition pruning
- Distributed architecture with horizontal scalability

Both databases contain identical data but are optimized for different query patterns.